In [1]:
%%capture

import warnings
# suppress user warnings during execution
warnings.filterwarnings(action='ignore', category=UserWarning)

%pip install --upgrade pip
%pip install spacy
%pip install ipywidgets
#%pip install -U jupyter
%sx python -m spacy download en_core_web_sm
%sx python -m spacy download fr_core_news_sm

# suppress user warnings during execution
warnings.filterwarnings(action='ignore', category=UserWarning)


def get_configured_pipeline(language: str="en"):
    clean_language = language.lower().strip()[:2]
    # use a predefined spaCy pipeline, disabling the default NER component
    packages = {
        "en": "en_core_web_sm",
        "fr": "fr_core_news_sm"
    }
    authorities = {
        "en": "p0kh9ds", # Perio.do authority for English periods
        "fr": "p02chr4" # Perio.do authority for French periods
    }
    clean_language = language.strip().lower()[:2]
    package_name: str|None = packages.get(clean_language, None)
    authority_name: str|None = authorities.get(clean_language, None)
    if package_name is None or authority_name is None:
        raise ValueError(f"Unsupported language code \"{language}\"")

    nlp = spacy.load(package_name, disable = ['ner'])
        
    # add rematch2 periodo_ruler component to the end of the pipeline
    # looking for terms from Perio.do authority "p0kh9ds" (http://purl.org/heritagedata/schemes/eh_period)
    nlp.add_pipe("yearspan_ruler", last=True)
    nlp.add_pipe("periodo_ruler", config={"periodo_authority_id": authority_name}, last=True)
    nlp.add_pipe("child_span_remover", last=True)
    return nlp

## Introduction
This notebook demonstrates the use of custom miltilingual pipeline component to identify temporal entites in free text, with working examples of usage. The following example python script uses the ``yearspan_ruler`` and ``periodo_ruler`` components to identify year span expressions (e.g. "Early 1900s") and named periods (terms from a specified Perio.do authority) in a given text. The output is then displayed as both inline HTML markup and as a table.

In [2]:
# Using pipeline components
import spacy
from components import DocSummary, child_span_remover
from IPython.display import display, HTML

# use a predefined spaCy pipeline, disabling the default NER component
nlp = get_configured_pipeline("en")

# processing example text using the modified pipeline
# (text from https://doi.org/10.5284/1100106)
txt = '''
Eight trenches were excavated across the 0.93ha Site. Previous archaeological works immediately to the north of the \
Site in the early 1990s, including field walking and targeted excavation, recorded worked flint and a pit possibly dated to \
the Late Iron Age, while other investigations in the wider area have identified remains of Late Bronze Age, Early Iron Age and \
Saxon date. Evidence for truncation of the deposit sequence was noted across the development area, with the Site area potentially \
having been levelled and drained prior to the establishment of turf pitches, with topsoil present only as a very thin turf horizon overlying \
sterile subsoil and brickearth deposits. The evaluation identified a single pit of Late Bronze Age to Early Iron Age date. No other \
archaeological features were located during the works.
'''
doc = nlp(txt)

# display highlighted entities in context
summary = DocSummary(doc)
html = summary.doctext_to_html()
display(HTML(html))
txt = summary.spans_to_text()
print(txt)

 start  end  token_start  token_end    label                                    id            text  sec_score sections  sig_proximity  score score_explain context
   129  139           23         24 YEARSPAN                                           early 1990s        0.0                     0.0    0.0                      
   245  257           43         45   PERIOD http://n2t.net/ark:/99152/p0kh9ds8p8k   Late Iron Age        0.0                     0.0    0.0                      
   332  346           58         60   PERIOD http://n2t.net/ark:/99152/p0kh9dsqkbq Late Bronze Age        0.0                     0.0    0.0                      
   349  362           62         64   PERIOD http://n2t.net/ark:/99152/p0kh9dszskn  Early Iron Age        0.0                     0.0    0.0                      
   722  736          125        127   PERIOD http://n2t.net/ark:/99152/p0kh9dsqkbq Late Bronze Age        0.0                     0.0    0.0                      
   741  754          1

The pipeline components contain language-specific patterns to facilitate similar processing in other languages. The following example uses the same ``yearspan_ruler`` and ``periodo_ruler`` components with a French IE pipeline and a specific Perio.do authority file identifier (see http://n2t.net/ark:/99152/p02chr4 - "PACTOLS chronology periods used in DOLIA data, 2021") to apply the same functionality on an extract of a French document.

In [3]:
# Using rematch2 components in another language
import spacy
from components import DocSummary, child_span_remover
from IPython.display import display, HTML

# use predefined (French) spaCy pipeline, disabling the default NER component
nlp = get_configured_pipeline("fr")
# process example text using the modified pipeline 
doc = nlp("Quelques éléments lithiques Würm IV dispersés sont des indicateurs de la période néolithique d'environ fin 11000 - début 10000 av JC. Un ensemble de fosses, circonscrit sur une surface de 35 m2, a livré du mobilier céramique du premier âge du Fer (Hallstatt C) et quelques éléments lithiques. Un bâtiment sur poteaux se situe à une distance de 100 m vers l’ouest. Pour la période de La Tène finale et gallo-romaine, l’ensemble des vestiges fossoyés et bâtis sont concentrés sur une parcelle à la croisée de deux chemins actuels présents sur le cadastre de 1807. L’élément structurant majeur est le fossé F6 qui traverse perpendiculairement la parcelle, large de 3 m pour une profondeur de 1,80 m sous la terre arable. Seul un angle de fossé, de nature différente et de plus petite taille, semble participer à cette même organisation du paysage. Un bâtiment de plan rectangulaire, 13 x 9 m, sur fondations de schiste dont deux angles ont été découverts, est orienté de façon identique au fossé F6. Il est très bien fondé sur une profondeur de 0,70 m avec de gros blocs de schiste. Si quelques rares tessons du Haut-Empire ont été trouvés dans la fondation du bâtiment, le mobilier céramique du colmatage de la zone humide située à 10 m au sud-ouest, est compris entre le milieu du Ier siècle et le début du IIe siècle de notre ère. Situé à 45 m plus au sud et parallèle au bâtiment, un fossé rectiligne de 3 m de large pour 1,80 m de profondeur sous la terre arable, scinde l’espace en deux. Son creusement en V à été comblé en deux temps. La première phase est une sédimentation naturelle qui a piégé quelques tessons protohistoriques et des scories ferreuses dont un culot de forge. Le colmatage supérieur est composé de matériaux issus de la démolition avec de très nombreuses tuiles, des blocs de schiste brut ainsi que quelques tessons de céramique gallo-romaine. De nombreux trous de poteaux et fosses se situent entre ces deux structures majeures. Un angle de fossé dessinant l’amorce d’un enclos se développe au sud. Du mobilier La Tène finale a également été trouvé dans une fosse située dans cette espace. Un réseau fossoyé se développe à l’est, très érodé du côté nord. Des fragments de céramique possiblement haut Moyen Âge ont été trouvés dans son comblement")
# display highlighted entities in context
summary = DocSummary(doc)
html = summary.doctext_to_html()
display(HTML(html))
txt = summary.spans_to_text()
print(txt)

 start  end  token_start  token_end    label                                    id                    text  sec_score sections  sig_proximity  score score_explain context
    28   34            3          4   PERIOD http://n2t.net/ark:/99152/p02chr4gqd2                 Würm IV        0.0                     0.0    0.0                      
    81   91           12         12   PERIOD http://n2t.net/ark:/99152/p02chr4ghc6             néolithique        0.0                     0.0    0.0                      
   121  131           19         21 YEARSPAN                                                   10000 av JC        0.0                     0.0    0.0                      
   228  245           42         45   PERIOD http://n2t.net/ark:/99152/p02chr4rx4z      premier âge du Fer        0.0                     0.0    0.0                      
   248  256           47         47   PERIOD http://n2t.net/ark:/99152/p02chr4n5w5               Hallstatt        0.0                     0.0    

Other vocabulary-based components are included to identify terms from specific controlled vocabularies. The following example identifies terms originating from the Getty Art & Architecture Thesaurus and from the UK FISH cultural heritage vocabularies. As these are English language vocabularies they are used in association with an English IE pipeline.

In [4]:
# Using vocabulary pipeline components
import spacy, json
from components.VocabularyRuler import *
from IPython.display import display, HTML

txt = """
This collection comprises site data (images, a report, a project database and GIS data) from an archaeological excavation undertaken by Cotswold Archaeology between January and February 2020 at Lydney B Phase III, Archers Walk, Lydney, Gloucestershire. An area of 0.6ha was excavated within this phase (Phase III) of a wider development area.
Aside from three residual flints, none closely datable, the earliest remains comprised a small assemblage of Roman pottery and ceramic building material, also residual and most likely derived from a Roman farmstead found immediately to the north within the Phase II excavation area. A single sherd of Anglo-Saxon grass-tempered pottery was also residual.
The earliest features, which accounted for the majority of the remains on site, relate to medieval agricultural activity focused within a large enclosure. There was little to suggest domestic occupation within the site: the pottery assemblage was modest and well abraded, whilst charred plant remains were sparse, and, as with some metallurgical residues, point to waste disposal rather than the locations of processing or consumption. A focus of occupation within the Rodley Manor site, on higher ground 160m to the north-west, seems likely, with the currently site having lain beyond this and providing agricultural facilities, most likely corrals and pens for livestock. Animal bone was absent, but the damp, low-lying ground would have been best suited to cattle. An assemblage of medieval coins recovered from the subsoil during a metal detector survey may represent a dispersed hoard.
"""

# create pipeline and add one or more custom pipeline components
nlp = spacy.load("en_core_web_sm", disable=['ner'])

nlp.add_pipe(
    "vocabulary_ruler", 
    name = "object_types_ruler",
    last = True, 
    config = {
        "default_label": "FISH_OBJECT",
        "token_pos": ["NOUN"],
        "lemmatize": True,
        "min_lemm_length": 3,
        "min_term_length": 3,
        "patt_list": json.load(open("./vocabularies/patterns_FISH_mda_obj_20260513.json")),
        "supp_list": json.load(open("./vocabularies/supp_list_FISH_ARCHOBJECTS.json")), 
        "stop_list": json.load(open("./vocabularies/stop_list_FISH_ARCHOBJECTS.json"))
     }
) 

nlp.add_pipe("child_span_remover", last=True)

# process example text using the modified pipeline
doc = nlp(txt)

# display highlighted entities in context
summary = DocSummary(doc)
html = summary.doctext_to_html()
display(HTML(html))
txt = summary.spans_to_text()
print(txt)


 start  end  token_start  token_end       label                                                           id                      text  sec_score sections  sig_proximity  score score_explain context
   370  375           70         70 FISH_OBJECT  http://purl.org/heritagedata/schemes/mda_obj/concepts/97566                    flints        0.0                     0.0    0.0                      
   459  465           85         85 FISH_OBJECT http://purl.org/heritagedata/schemes/mda_obj/concepts/137051                   pottery        0.0                     0.0    0.0                      
   471  495           87         89 FISH_OBJECT http://purl.org/heritagedata/schemes/mda_obj/concepts/141190 ceramic building material        0.0                     0.0    0.0                      
   636  640          115        115 FISH_OBJECT http://purl.org/heritagedata/schemes/mda_obj/concepts/137051                     sherd        0.0                     0.0    0.0                      
   67